# Consent-Based Voice Adaptation

This notebook prepares a pretrained text-to-speech model for a consented teacher voice adaptation workflow.

What this notebook does:
- Validates consented speech recordings and transcripts
- Loads a pretrained TTS base model
- Applies a lightweight LoRA adapter for CPU-friendly fine-tuning
- Extracts a reference speaker embedding from consented audio
- Saves adapter and configuration artifacts for later inference

What this notebook does not do:
- Train a voice model from scratch
- Work without a consented dataset

Expected inputs:
- `voice_data/manifest.jsonl` with `audio_path` and `text` fields
- Clean WAV recordings of the teacher with matching transcripts

Expected outputs:
- `voice_model_artifacts/teacher_voice_lora_adapter`
- `voice_model_artifacts/teacher_voice_package.pt`
- `voice_model_artifacts/train_config.json`

In [171]:
import gc
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import librosa
import numpy as np
import torch
from torch.utils.data import Dataset

# TTS imports - Coqui TTS adaptation path
from TTS.api import TTS

In [172]:
import os
os.getcwd()
#os.chdir('/mnt/c/Users/xu6189ne/OneDrive - Minnesota State/Documents/CLASSES/business_programming/projects/big_project')
os.chdir('/mnt/d/my_repos/college/business_programming/big_project')
os.getcwd()

'/mnt/d/my_repos/college/business_programming/big_project'

## presets

In [173]:
# Paths and artifact names
PROJECT_ROOT_DEFAULT = "."
MANIFEST_REL_PATH = "voice_data/manifest.jsonl"
OUTPUT_DIR_NAME = "voice_model_artifacts"
OUTPUT_WAV_DIR_NAME = "synthesized_audio"  # Subdirectory for .wav output files
ADAPTER_DIR_NAME = "teacher_voice_lora_adapter"
PACKAGE_FILENAME = "teacher_voice_package.pt"
TRAIN_CONFIG_FILENAME = "train_config.json"
PROCESSOR_DIR_NAME = "processor"
VOCODER_DIR_NAME = "vocoder"

# Model and runtime defaults
BASE_MODEL_NAME = "tts_models/en/ljspeech/glow-tts"  # Coqui TTS female voice model
VOCODER_MODEL_NAME = "vocoder_models/en/ljspeech/hifigan_v2"  # Coqui vocoder
SPEAKER_ENCODER_NAME = "speechbrain/spkrec-ecapa-voxceleb"
DEVICE_NAME = "cpu"
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# Reused numeric constants
MFCC_DIMENSIONS = 40
NORM_EPSILON = 1e-8
MAX_REFERENCE_ROWS = 5

## configuration

In [174]:
@dataclass
class VoiceAdaptConfig:
    project_root: str = PROJECT_ROOT_DEFAULT
    manifest_path: str = MANIFEST_REL_PATH
    output_dir: str = OUTPUT_DIR_NAME
    output_wav_dir: str = OUTPUT_WAV_DIR_NAME  # Subdirectory for synthesized audio

    base_model_name: str = BASE_MODEL_NAME
    vocoder_name: str = VOCODER_MODEL_NAME
    speaker_encoder_name: str = SPEAKER_ENCODER_NAME

    sample_rate: int = 16000
    max_audio_seconds: float = 8.0
    max_text_length: int = 256

    batch_size: int = 1
    grad_accum_steps: int = 4
    learning_rate: float = 1e-4
    num_epochs: int = 3
    seed: int = 42
    cpu_threads: int = 4

    # Coqui TTS fine-tuning parameters
    use_speaker_embedding: bool = True
    num_mels: int = 80
    num_freq: int = 1025


cfg = VoiceAdaptConfig()
PROJECT_ROOT = Path(cfg.project_root).resolve()
MANIFEST_PATH = (PROJECT_ROOT / cfg.manifest_path).resolve()
OUTPUT_DIR = (PROJECT_ROOT / cfg.output_dir).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_WAV_DIR = OUTPUT_DIR / cfg.output_wav_dir
OUTPUT_WAV_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(DEVICE_NAME)
torch.set_num_threads(cfg.cpu_threads)
torch.manual_seed(cfg.seed)
random.seed(cfg.seed)

{
    "manifest": str(MANIFEST_PATH),
    "output_dir": str(OUTPUT_DIR),
    "output_wav_dir": str(OUTPUT_WAV_DIR),
    "device": str(DEVICE),
    "base_model": cfg.base_model_name,
    "vocoder": cfg.vocoder_name,
}

{'manifest': '/mnt/d/my_repos/college/business_programming/big_project/voice_data/manifest.jsonl',
 'output_dir': '/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts',
 'output_wav_dir': '/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/synthesized_audio',
 'device': 'cpu',
 'base_model': 'tts_models/en/ljspeech/glow-tts',
 'vocoder': 'vocoder_models/en/ljspeech/hifigan_v2'}

## functions

### normalize_text
Standardizes transcript text by trimming whitespace, lowering case, and collapsing duplicate spaces.

In [175]:
def normalize_text(text: str) -> str:
    return " ".join(text.strip().lower().split())

### load_manifest
Reads and validates the JSONL manifest, resolves audio paths, and normalizes transcript text.

In [176]:
def load_manifest(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(
            f"Manifest not found at {path}. Create JSONL with keys: audio_path, text"
        )

    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            if "audio_path" not in item or "text" not in item:
                raise ValueError("Each manifest row needs 'audio_path' and 'text'.")

            audio_path = Path(item["audio_path"])
            resolved_audio = (
                str((path.parent / audio_path).resolve())
                if not audio_path.is_absolute()
                else str(audio_path)
            )

            rows.append(
                {
                    "audio_path": resolved_audio,
                    "text": normalize_text(item["text"]),
                }
            )

    if not rows:
        raise ValueError("Manifest is empty.")

    return rows

### build_dataset_summary
Computes quick metadata about the loaded training rows for sanity checking.

In [177]:
def build_dataset_summary(rows: list[dict[str, str]]) -> dict[str, Any]:
    return {
        "num_examples": len(rows),
        "avg_text_length": float(np.mean([len(row["text"]) for row in rows])) if rows else 0.0,
        "sample_audio_paths": [row["audio_path"] for row in rows[:3]],
    }

### load_audio_mono
Loads mono audio at a fixed sample rate and trims it to a configured maximum duration.

In [178]:
def load_audio_mono(audio_path: Path, sample_rate: int, max_audio_seconds: float) -> np.ndarray:
    waveform, _ = librosa.load(audio_path, sr=sample_rate, mono=True)
    max_samples = int(sample_rate * max_audio_seconds)
    return waveform[:max_samples]

### TeacherVoiceDataset
Wraps manifest rows as dataset items with waveform tensors and metadata.

In [179]:
class TeacherVoiceDataset(Dataset):
    def __init__(self, rows: list[dict[str, str]], cfg: VoiceAdaptConfig):
        self.rows = rows
        self.cfg = cfg

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.rows[idx]
        audio_path = Path(row["audio_path"])
        waveform = load_audio_mono(audio_path, self.cfg.sample_rate, self.cfg.max_audio_seconds)

        return {
            "text": row["text"],
            "audio_path": str(audio_path),
            "waveform": torch.tensor(waveform, dtype=torch.float32),
            "duration_seconds": float(waveform.shape[0] / self.cfg.sample_rate),
        }

### collate_teacher_batch
Pass-through collate function for dataloader compatibility.

In [180]:
def collate_teacher_batch(batch: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return batch

### _extract_mfcc_embedding
Builds a lightweight speaker embedding from MFCC statistics.

In [181]:
def _extract_mfcc_embedding(audio_path: str, cfg: VoiceAdaptConfig) -> torch.Tensor:
    """Create a lightweight CPU speaker representation from MFCC statistics."""
    waveform = load_audio_mono(Path(audio_path), cfg.sample_rate, cfg.max_audio_seconds)
    mfcc = librosa.feature.mfcc(
        y=waveform,
        sr=cfg.sample_rate,
        n_mfcc=MFCC_DIMENSIONS,
    )
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    feat = np.concatenate(
        [
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            delta.mean(axis=1),
            delta.std(axis=1),
            delta2.mean(axis=1),
            delta2.std(axis=1),
        ]
    )
    embedding = torch.tensor(feat, dtype=torch.float32)

    target_dim = 512
    if embedding.numel() < target_dim:
        embedding = torch.nn.functional.pad(embedding, (0, target_dim - embedding.numel()))
    elif embedding.numel() > target_dim:
        embedding = embedding[:target_dim]

    embedding = embedding / embedding.norm(p=2).clamp(min=NORM_EPSILON)
    return embedding

### extract_speaker_embedding
Aggregates per-recording embeddings into one normalized speaker embedding.

In [182]:
def extract_speaker_embedding(reference_rows: list[dict[str, str]], cfg: VoiceAdaptConfig) -> torch.Tensor:
    if not reference_rows:
        raise ValueError("At least one reference recording is required for the speaker embedding.")

    embeddings = [_extract_mfcc_embedding(row["audio_path"], cfg) for row in reference_rows]
    speaker_embedding = torch.stack(embeddings).mean(dim=0)
    speaker_embedding = speaker_embedding / speaker_embedding.norm(p=2).clamp(min=NORM_EPSILON)
    return speaker_embedding

### Coqui TTS Model Loading & Fine-tuning
Functions for loading and training with Coqui TTS instead of SpeechT5.

In [183]:
def load_coqui_model(cfg: VoiceAdaptConfig):
    """Load Coqui TTS model for synthesis and adaptation workflows."""
    try:
        # Coqui TTS high-level API exposes synthesis via tts.tts(...) / tts.tts_to_file(...)
        tts = TTS(model_name=cfg.base_model_name, gpu=False)
        synthesizer = getattr(tts, "synthesizer", None)
        return tts, synthesizer
    except Exception as e:
        raise RuntimeError(f"Failed to load Coqui TTS model {cfg.base_model_name}: {e}")

In [184]:
def prepare_coqui_dataset(rows: list[dict[str, str]], cfg: VoiceAdaptConfig) -> dict[str, Any]:
    """Prepare dataset in Coqui format for training."""
    metadata = []
    
    for row in rows:
        audio_path = Path(row["audio_path"])
        if not audio_path.exists():
            print(f"Warning: Audio file not found: {audio_path}")
            continue
        
        # Coqui metadata format: audio_path|speaker_id|language|text
        metadata.append({
            "audio_path": str(audio_path),
            "speaker_name": "target_speaker",
            "language": "en",
            "text": row["text"]
        })
    
    if not metadata:
        raise ValueError("No valid audio files found for training")
    
    return metadata

In [185]:
def finetune_coqui_model(
    dataset_metadata: list[dict[str, str]],
    cfg: VoiceAdaptConfig,
) -> dict[str, Any]:
    """
    Prepare fine-tuning assets for Coqui Trainer.

    Note:
    The high-level TTS API (TTS(...).tts) is for inference. Reliable fine-tuning is done
    through Coqui's Trainer pipeline. This function creates trainer-ready metadata and
    returns a command scaffold to run fine-tuning externally.
    """
    metadata_file = OUTPUT_DIR / "coqui_finetune_metadata.csv"

    # Coqui single-speaker metadata format: audio_file|text
    with metadata_file.open("w", encoding="utf-8") as f:
        for item in dataset_metadata:
            f.write(f"{item['audio_path']}|{item['text']}\n")

    trainer_output_dir = OUTPUT_DIR / "coqui_finetuned"
    trainer_output_dir.mkdir(parents=True, exist_ok=True)

    trainer_note = (
        "Use Coqui Trainer with a model-specific config for GlowTTS/VITS/XTTS fine-tuning. "
        "This notebook prepared metadata, but training should be launched via trainer CLI/script."
    )

    return {
        "status": "prepared_for_trainer",
        "samples": len(dataset_metadata),
        "metadata_file": str(metadata_file),
        "trainer_output_dir": str(trainer_output_dir),
        "note": trainer_note,
    }

In [186]:
def save_coqui_artifacts(
    tts,
    cfg: VoiceAdaptConfig,
    training_info: dict[str, Any],
) -> dict[str, str]:
    """Save Coqui TTS model artifacts."""
    
    artifacts_paths = {}
    
    try:
        # Save model checkpoint
        model_dir = OUTPUT_DIR / "coqui_model_checkpoint"
        model_dir.mkdir(parents=True, exist_ok=True)
        
        # Coqui saves its own checkpoints, just track the path
        artifacts_paths["model_dir"] = str(model_dir)
        artifacts_paths["config_path"] = str(OUTPUT_DIR / TRAIN_CONFIG_FILENAME)
        artifacts_paths["training_info"] = str(OUTPUT_DIR / "coqui_training_info.json")
        
        # Save configuration
        with open(OUTPUT_DIR / TRAIN_CONFIG_FILENAME, "w", encoding="utf-8") as f:
            json.dump(asdict(cfg), f, indent=2)
        
        # Save training information
        with open(OUTPUT_DIR / "coqui_training_info.json", "w", encoding="utf-8") as f:
            json.dump(training_info, f, indent=2)
        
        print(f"Artifacts saved to {OUTPUT_DIR}")
        
    except Exception as e:
        print(f"Warning: Could not save all artifacts: {e}")
    
    return artifacts_paths

## Training with Coqui TTS

In [187]:
# Load training data
rows = load_manifest(MANIFEST_PATH)
summary = build_dataset_summary(rows)

if len(rows) == 0:
    raise ValueError("No training data found in the manifest.")

print(f"Loaded {len(rows)} training samples")

# Load Coqui TTS model
print(f"Loading Coqui TTS model: {cfg.base_model_name}")
tts, synthesizer = load_coqui_model(cfg)

# Prepare dataset for Coqui fine-tuning
coqui_metadata = prepare_coqui_dataset(rows, cfg)

# Prepare trainer assets (no crash, trainer-ready output)
print("Preparing Coqui trainer assets...")
training_info = finetune_coqui_model(coqui_metadata, cfg)

# Save artifacts
artifacts = save_coqui_artifacts(tts, cfg, training_info)

{
    "dataset_summary": summary,
    "training_info": training_info,
    "artifacts": artifacts,
}

Loaded 120 training samples
Loading Coqui TTS model: tts_models/en/ljspeech/glow-tts
 > tts_models/en/ljspeech/glow-tts is already downloaded.
 > vocoder_models/en/ljspeech/multiband-melgan is already downloaded.
 > Using model: glow_tts
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:0
 | > fft_size:1024
 | > power:1.1
 | > preemphasis:0.0
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:True
 | > mel_fmin:50.0
 | > mel_fmax:7600.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:1.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:True
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Vocoder Model

{'dataset_summary': {'num_examples': 120,
  'avg_text_length': 208.28333333333333,
  'sample_audio_paths': ['/mnt/d/my_repos/college/business_programming/big_project/voice_data/recordings/1.m4a',
   '/mnt/d/my_repos/college/business_programming/big_project/voice_data/recordings/2.wav',
   '/mnt/d/my_repos/college/business_programming/big_project/voice_data/recordings/3.wav']},
 'training_info': {'status': 'prepared_for_trainer',
  'samples': 1,
  'metadata_file': '/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/coqui_finetune_metadata.csv',
  'trainer_output_dir': '/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/coqui_finetuned',
  'note': 'Use Coqui Trainer with a model-specific config for GlowTTS/VITS/XTTS fine-tuning. This notebook prepared metadata, but training should be launched via trainer CLI/script.'},
 'artifacts': {'model_dir': '/mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/coqui_m

## Synthesizing Speech with Coqui TTS

In [192]:
import scipy.io.wavfile as wavfile
from pathlib import Path

# Text to synthesize
text = "hello world"

# Generate speech using Coqui TTS
# The TTS model handles both text processing and vocoding internally
print(f"Synthesizing: '{text}'")

try:
    # Use the TTS model to generate speech
    # Coqui returns audio directly
    wav = tts.tts(text=text, speaker_idx=None)
    
    # Convert to numpy array if needed
    if isinstance(wav, torch.Tensor):
        waveform_np = wav.cpu().numpy()
    else:
        waveform_np = np.array(wav)
    
    # Ensure mono and correct shape
    if len(waveform_np.shape) > 1:
        waveform_np = waveform_np.squeeze()
    
    # Save as WAV file to dedicated output directory
    output_path = OUTPUT_WAV_DIR / f"output_{text.replace(' ', '_')}.wav"
    
    # Normalize and convert to int16
    if waveform_np.max() <= 1.0:
        audio_int16 = (waveform_np * 32767).astype('int16')
    else:
        audio_int16 = (waveform_np / waveform_np.max() * 32767).astype('int16')
    
    wavfile.write(str(output_path), cfg.sample_rate, audio_int16)
    
    print(f"Audio file saved to: {output_path}")
    print(f"Sample rate: {cfg.sample_rate} Hz")
    print(f"Duration: {len(waveform_np) / cfg.sample_rate:.2f} seconds")
    
except Exception as e:
    print(f"Error during synthesis: {e}")
    print("Make sure the model is properly trained/loaded")

Synthesizing: 'hello world'
 > Text splitted to sentences.
['hello world']
 > Processing time: 0.06612586975097656
 > Real-time factor: 0.04783712034150371
Audio file saved to: /mnt/d/my_repos/college/business_programming/big_project/voice_model_artifacts/synthesized_audio/output_hello_world.wav
Sample rate: 16000 Hz
Duration: 1.91 seconds
